In [1]:
import json
from Pipeline import GeoTKGPipeline

with open("D:\\GeoTKG\\cleandata\\tie\\test.json", "r") as f:
    examples=[json.loads(line) for line in f]
preds = []
process_sep = []
model = GeoTKGPipeline()
batch_size = 4
for i in range(0, len(examples), batch_size):
    samples = examples[i:i+batch_size]
    for j, sample in enumerate(samples):
        if len(sample['text'])>50:
            process_sep.append(examples.index(sample))
            samples.pop(j)
    dcts = [inst['value'] for sample in samples for inst in sample['instances'] if inst['type'] != "EVENT" and inst['id'] == 0]
    text = [" ".join([wrd for sent in sample['text'] for wrd in sent]) for sample in samples]
    output = model.pred(text, dcts)
    preds.extend(output)
    print(f"Processed {batch_size+i}/{len(examples)}")

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processed 4/602
Processed 8/602
Processed 12/602
Processed 16/602
Processed 20/602
Processed 24/602
Processed 28/602
Processed 32/602
Processed 36/602
Processed 40/602
Processed 44/602
Processed 48/602
Processed 52/602
Processed 56/602
Processed 60/602
Processed 64/602
Processed 68/602
Processed 72/602
Processed 76/602
Processed 80/602
Processed 84/602
Processed 88/602
Processed 92/602
Processed 96/602
Processed 100/602
Processed 104/602
Processed 108/602
Processed 112/602
Processed 116/602
Processed 120/602
Processed 124/602
Processed 128/602
Processed 132/602
Processed 136/602
Processed 140/602
Processed 144/602
Processed 148/602
Processed 152/602
Processed 156/602
Processed 160/602
Processed 164/602
Processed 168/602
Processed 172/602
Processed 176/602
Processed 180/602
Processed 184/602
Processed 188/602
Processed 192/602
Processed 196/602
Processed 200/602
Processed 204/602
Processed 208/602
Processed 212/602
Processed 216/602
Processed 220/602
Processed 224/602
Processed 228/602


In [2]:
process_sep
for pred_i in process_sep:
    dcts = [inst['value'] for inst in examples[pred_i]['instances'] if inst['type'] != "EVENT" and inst['id'] == 0]
    text = [" ".join([wrd for sent in examples[pred_i]['text'] for wrd in sent])]
    output = model.pred(text, dcts)
    preds.insert(pred_i, output[0])

In [45]:
len(preds)

602

In [4]:
geotkg_preds = []
for fn, (quins, trips) in enumerate(preds):
    geotkg_preds.append({"quintuples":quins, "triples":trips})

In [54]:
# with open("GeoTKG-TIE-test-preds.json", 'w') as json_file:
#     for sample in preds:
#         json_file.write(json.dumps(sample)+"\n")
from datetime import datetime, date

INVERSE = {
    "AFTER": "BEFORE",
    "BEFORE": "AFTER",
    "CONTAINS": "DURING",
    "DURING": "CONTAINS",
    "EQUALS": "EQUALS",
    "OVERLAPS": "OVERLAPS",       # add if you use these
    "IDENTITY": "IDENTITY"
    # extend as needed
}

def get_data(path, preprocessor=None):
    with open(path, "r") as f:
        examples=[json.loads(line) for line in f]
    if preprocessor is not None:
        examples = [preprocessor(example) for example in examples]
    return examples

class TempRelObj:
    def __init__(self, temprel_list, exhaustive=False):
        self.all_pairs = []
        for triple in temprel_list:
            try:
                e1, rel, e2 = triple["e1"], triple['rel'], triple['e2']
            except:
                e1, rel, e2 = triple[0], triple[1], triple[2]
            self.all_pairs.append([e1, rel, e2])
        if exhaustive:
            for (e1, rel, e2) in self.all_pairs:
                c = self.all_pairs.count([e2, INVERSE[rel], e1])
                if c == 0:
                    self.all_pairs.append([e2, INVERSE[rel], e1])
            for trip1 in self.all_pairs:
                for trip2 in self.all_pairs:
                    if trip1 == trip2 or trip1[1] != trip2[1] or trip1[2] != trip2[0]:
                        continue
                    if trip1[1] == "BEFORE" and trip2[1] == "BEFORE":
                        if self.all_pairs.count([trip1[0], "BEFORE", trip2[2]]) == 0:
                            self.all_pairs.append([trip1[0], "BEFORE", trip2[2]])
                    if trip1[1] == "AFTER" and trip2[1] == "AFTER":
                        if self.all_pairs.count([trip1[0], "AFTER", trip2[2]]) == 0:
                            self.all_pairs.append([trip1[0], "AFTER", trip2[2]])
                    if trip1[1] == "CONTAINS" and trip2[1] == "CONTAINS":
                        if self.all_pairs.count([trip1[0], "CONTAINS", trip2[2]]) == 0:
                            self.all_pairs.append([trip1[0], "CONTAINS", trip2[2]])
                    if trip1[1] == "DURING" and trip2[1] == "DURING":
                        if self.all_pairs.count([trip1[0], "DURING", trip2[2]]) == 0:
                            self.all_pairs.append([trip1[0], "DURING", trip2[2]])
                    if trip1[1] == "EQUALS" and trip2[1] == "EQUALS":
                        if self.all_pairs.count([trip1[0], "EQUALS", trip2[2]]) == 0:
                            self.all_pairs.append([trip1[0], "EQUALS", trip2[2]])
                


import isodate
def gentext_to_iso8601(gentext: str):
    parsers = {
        isodate.parse_date:"DATE",
        isodate.parse_time:"TIME",
        isodate.parse_datetime:"TIME",
        isodate.parse_duration:"DURATION",
        isodate.parse_tzinfo:"SET",
    }

    for parser in parsers:
        try:
            out = parser(gentext)
            type_ = parsers[parser]
            return out, type_
        except Exception:
            continue

    # If none of the parsers worked
    #print(f"UNREC: {gentext}")
    return None

def get_start_end_times(event_times: list, dct):
    all_times = [gentext_to_iso8601(time) for time in event_times]
    if len(all_times) == 0:
        return None, None
    dates = [time[0] for time in all_times if time[1] == "DATE"]
    times = [time[0] for time in all_times if time[1] == "TIME"]
    durs = [time[0] for time in all_times if time[1] == "DURATION"]

    if len(dates)>0:
        s_time = min(dates)
        e_time = max(dates)
    elif len(times)>0:
        s_time = min(times)
        e_time = max(times)
    else:
        s_time = dct
        e_time = dct
    
    for time in times:
        value = time
        if type(s_time) == date:
            value = time.date()
        if s_time >= value:
            s_time = time
        elif e_time <= value:
            e_time = time
    
    s_time = datetime.combine(s_time, datetime.min.time()) if type(s_time)==date else s_time
    e_time = datetime.combine(e_time, datetime.min.time()) if type(e_time)==date else e_time

    for duration in durs:
        if s_time + duration > e_time:
            e_time = s_time + duration

    return s_time, e_time

import datetime as _dt
from typing import List, Tuple, Optional
from copy import deepcopy

def _cmp_date_components(gold, pred) -> bool:
    """Any of year/month/day matches."""
    g = {"y": getattr(gold, "year", None), "m": getattr(gold, "month", None), "d": getattr(gold, "day", None)}
    p = {"y": getattr(pred, "year", None), "m": getattr(pred, "month", None), "d": getattr(pred, "day", None)}
    return any(g[k] is not None and g[k] == p[k] for k in ("y", "m", "d"))

def _cmp_time_components(gold, pred) -> bool:
    """Any of hour/minute/second matches (ignores microseconds)."""
    g = {"h": getattr(gold, "hour", None), "m": getattr(gold, "minute", None), "s": getattr(gold, "second", None)}
    p = {"h": getattr(pred, "hour", None), "m": getattr(pred, "minute", None), "s": getattr(pred, "second", None)}
    return any(g[k] is not None and g[k] == p[k] for k in ("h", "m", "s"))

def _cmp_datetime_components(gold, pred) -> bool:
    """Any date *or* time component matches."""
    date_ok = _cmp_date_components(gold, pred)
    time_ok = _cmp_time_components(gold, pred)
    return date_ok or time_ok

def _total_seconds(x) -> Optional[float]:
    # isodate durations often become datetime.timedelta
    if isinstance(x, _dt.timedelta):
        return x.total_seconds()
    return None

def _cmp_duration_any_component(gold, pred) -> bool:
    """
    Mark correct if total seconds equal (most practical),
    OR if both encode at least one matching component (days/hours/minutes/seconds) when derivable.
    """
    gs = _total_seconds(gold)
    ps = _total_seconds(pred)
    if gs is not None and ps is not None:
        return abs(gs - ps) < 1e-6

    # Fallback: try to infer rough components if timedelta-like but not precise
    # (Most libraries give timedelta; if not, we can’t safely decompose—return False.)
    return False

def _normalize_set_string(s: str) -> str:
    """
    Very light 'SET' normalization:
    - split common separators, strip whitespace, sort tokens, rejoin.
    Adjust to your dataset’s SET format.
    """
    for sep in [",", ";", "|"]:
        s = s.replace(sep, " ")
    toks = [t for t in s.split() if t]
    toks.sort()
    return " ".join(toks).lower()

def _cmp_set_relaxed(gold_str: str, pred_str: str) -> bool:
    """Any overlap in normalized token sets qualifies as relaxed-correct."""
    g = set(_normalize_set_string(gold_str).split())
    p = set(_normalize_set_string(pred_str).split())
    return len(g & p) > 0

def relaxed_correct_single(g: str, p: str) -> bool:
    """
    Strict equality first; if not equal, apply relaxed rule per ti_type.
    ti_type ∈ {"DATE","TIME","DATETIME","DURATION","SET"} (case-insensitive).
    """
    # Strict exact match first (you can move strict to your main metric if preferred)
    if g == p:
        return True
    # If parsing fails for either side, fall back to string-based relaxed checks for SET,
    # otherwise we can’t relax-match.
    if g is None and p is None:
        return True
    elif g is None or p is None:
        return False

    return _cmp_date_components(g, p) or _cmp_time_components(g, p) or _cmp_datetime_components(g, p)

def truth_quintuples_and_triples_preprocess(example):
    events = {}
    times = {}
    event_quins = {}
    ets = {}
    for instance in example['instances']:
        instance_id = instance["id"]
        if instance["type"] == "EVENT":
            events[instance_id] = instance
        else:
            times[instance_id] = instance

    triple_obj = TempRelObj(example['ee_temprels'])
    ee_trips = triple_obj.all_pairs
    for trip in ee_trips:
        trip[0] = events[trip[0]]['text']
        trip[2] = events[trip[2]]['text']

    for et in example["event_times"]:
        evid = et["event"] 
        if 'value' in times[et["time"]]:
            value = times[et["time"]]['value']
        else:
            value = None
        if evid not in ets:
            ets[evid] = [value]
        else:
            ets[evid].append(value)

    dct = gentext_to_iso8601(times[0]['value'])[0]

    for eid, event in events.items():
        s_time, e_time = get_start_end_times(ets.get(eid, []), dct)
        quint = {
            "event": event["text"],
            "subject": None,
            "object": None,
            "s_time": s_time,
            "e_time": e_time
        }
        event_quins[eid] = quint

    return {'times':list(times.values()), 'quintuples':event_quins, 'triples':list(ee_trips)}

def text_match(truth_text, pred_text):
    if pred_text is None and type(truth_text)==str:
        return False

    text_match = False
    if truth_text == pred_text:
        text_match = "strict"
    elif truth_text in pred_text or pred_text in truth_text:
        text_match = "relaxed"
    return text_match

def sample_quintuple_compare(truths, preds):
    preds_copy = deepcopy(preds)
    strict_results = []
    for truth in truths:
        matched = False
        for pred in preds_copy:
            if truth["event"]==pred["event"] and truth["s_time"]==pred["stime"] and truth["e_time"]==pred["e_time"]:
                strict_results.append(1)
                preds_copy.remove(pred)
                matched = True
                break
        if matched == False:
            strict_results.append(0)

    preds_copy = deepcopy(preds)
    relaxed_results = []
    for truth in truths:
        matched = False
        for pred in preds_copy:
            if text_match(truth["event"], pred['event'])!=False and relaxed_correct_single(truth["s_time"], pred['stime']) and relaxed_correct_single(truth["e_time"], pred['e_time']):
                relaxed_results.append(1)
                preds_copy.remove(pred)
                matched = True
                break
        if matched == False:
            relaxed_results.append(0)
    return strict_results, relaxed_results

In [55]:
test_data = get_data("D:\\GeoTKG\\cleandata\\tie\\test.json", truth_quintuples_and_triples_preprocess)


In [56]:
len(sample_quintuple_compare(list(test_data[0]['quintuples'].values()), geotkg_preds[0]['quintuples'])[0]), len(list(test_data[0]['quintuples'].values()))

(86, 86)

In [57]:
from sklearn.metrics import f1_score

strict_results = []
relaxed_results = []
for truth, pred in zip(test_data, geotkg_preds):
    strict_out, relaxed_out = sample_quintuple_compare(list(truth['quintuples'].values()), pred['quintuples'])
    relaxed_results.extend(relaxed_out)
    strict_results.extend(strict_out)

{"relaxed":f1_score([1]*len(relaxed_results), relaxed_results), "strict":f1_score([1]*len(strict_results), strict_results)}

{'relaxed': 0.6263263161463101, 'strict': 0.5795211553966417}

In [59]:
def sample_triple_compare(truths, preds):
    preds_copy = deepcopy(preds)
    preds_copy = TempRelObj(preds_copy, exhaustive=True).all_pairs
    results = []
    for truth in truths:
        matched = False
        for pred in preds_copy:
            if text_match(truth[0], pred[0])!=False and truth[1]==pred[1] and text_match(truth[2], pred[2])!=False:
                results.append(1)
                preds_copy.remove(pred)
                matched = True
                break
        if matched == False:
            results.append(0)
    return results

strict_results = []
for truth, pred in zip(test_data, geotkg_preds):
    #print(tkg_preds.index(pred))
    strict_out = sample_triple_compare(truth['triples'], 
                                       pred['triples'])
    strict_results.extend(strict_out)
f1_score([1]*len(strict_results), strict_results)

0.8627139991603303